In [2]:
import csv
import os
import glob
import sys
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split

CUDA_BIN = Path(r"C:/Program Files/NVIDIA GPU Computing Toolkit/CUDA/v11.5/bin")
CUDNN_BIN_CANDIDATES = [
    Path.cwd() / "third_party" / "cudnn-8.9.7-cuda11" / "bin",
    Path.cwd() / "cv_hands" / "third_party" / "cudnn-8.9.7-cuda11" / "bin",
]

dll_paths = []
if CUDA_BIN.exists():
    dll_paths.append(str(CUDA_BIN))

for candidate in CUDNN_BIN_CANDIDATES:
    if candidate.exists():
        dll_paths.append(str(candidate))
        break

if dll_paths:
    os.environ["PATH"] = ";".join(dll_paths + [os.environ.get("PATH", "")])

print("Python:", sys.executable)
print("CWD:", Path.cwd())
print("DLL search paths:", dll_paths)

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

RANDOM_SEED = 42

Python: d:\Profile\Documents\GitHub\FYP-SignLanguage\cv_hands\.venv-train310\Scripts\python.exe
CWD: d:\Profile\Documents\GitHub\FYP-SignLanguage\cv_hands
DLL search paths: ['C:\\Program Files\\NVIDIA GPU Computing Toolkit\\CUDA\\v11.5\\bin', 'd:\\Profile\\Documents\\GitHub\\FYP-SignLanguage\\cv_hands\\third_party\\cudnn-8.9.7-cuda11\\bin']
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Dataset Path

In [3]:
model_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.keras'
tflite_save_path = 'model/keypoint_classifier/keypoint_sequence_classifier.tflite'

# Parameters

In [4]:
SEQUENCE_LENGTH = 25
FEATURES_PER_FRAME = 80  # 42 hand + 10 face + 8 pose + 20 relative

# Load Dataset

In [5]:
# Load Dataset
csv_files = sorted(glob.glob('Words-Dataset/*_sequence.csv'))
X_sequences = []
y_sequences = []
for csv_file in csv_files:
    data = np.loadtxt(csv_file, delimiter=',', dtype='float32')
    X_sequences.append(data[:, 1:].reshape(-1, SEQUENCE_LENGTH, FEATURES_PER_FRAME))
    y_sequences.extend(data[:, 0])

X_dataset = np.concatenate(X_sequences, axis=0)
y_dataset = np.array(y_sequences)
y_dataset -= 1  # Convert from 1-based to 0-based indexing for TensorFlow

X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

# Load labels to determine number of classes
with open('Word-Label/keypoint_sequence_classifier_label.csv', encoding='utf-8-sig') as f:
    keypoint_sequence_classifier_labels = csv.reader(f)
    keypoint_sequence_classifier_labels = [row[0] for row in keypoint_sequence_classifier_labels]
NUM_CLASSES = len(keypoint_sequence_classifier_labels)
print(f"Number of classes: {NUM_CLASSES}")
print(f"Labels: {keypoint_sequence_classifier_labels}")

Number of classes: 152
Labels: ['你好', '學校', '同學', '屋企人', '鐘意', '唔鐘意', '點解', '彩虹', '謝謝', '等等', '對不起', '聾人', '我', '健聽', '\u2060手語', '現在', '高級', '認識/見面', '開心', '再見', '香港', '早餐', '護照', '護士', 'ok', '什麼', '麵', '紙巾', '有', '類別', '願望', '日本', '好', '懲罰', '講粗口', '人', '是', '不是', '需要', '不需要', '幫忙', '醫生', '咖啡', '想', '不想', '爸爸', '媽媽', '父母', '哥哥', '弟弟', '姐姐', '妹妹', '龍', '爺爺', '公公', '嫲嫲', '頭', '兒子', '女兒', '老公', '老婆', '頭痛', '頭盔', '我們', '鋼琴', '你', '喝', '句子', '學士', '衝突', '詞語', '工作', '標籤', '摩托車', '出糧', '噓', '厲害', '劍擊', '同事', '文職', '網絡', '運動', '學習', '緊張', '茶', '腹瀉', '義工', '功課', '零', '一', '二', '三', '四', '睡覺', '六', '七', '牛奶', '九', '助聽器', '盲', '水', '肚餓', '渴', '飽', '美味', '朋友', '愛', '介紹', '傷心', '生氣', '害怕', '累', '病', '痛', '舒服', '冷', '熱', '大', '小', '遠', '近', '快', '慢', '新', '舊', '平', '貴', '美麗', '聰明', '強壯', '弱', '左', '右', '上面', '下', '前', '後', '最鍾意', '關', '買', '教', '睇', '電話', '電影', '吃', '老師', '會去', '醫院', '假期', '責任', '一齊', '感覺']


# Build LSTM Model

In [6]:
if 'NUM_CLASSES' not in locals() or NUM_CLASSES == 0:
    print("No classes found. Please collect data first.")
else:
    from tensorflow.keras import layers
    model = tf.keras.models.Sequential([
        layers.Bidirectional(layers.LSTM(256, return_sequences=True), input_shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME)),
        layers.Dropout(0.3),
        layers.Bidirectional(layers.LSTM(128)),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])

    model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bidirectional (Bidirectiona  (None, 25, 512)          690176    
 l)                                                              
                                                                 
 dropout (Dropout)           (None, 25, 512)           0         
                                                                 
 bidirectional_1 (Bidirectio  (None, 256)              656384    
 nal)                                                            
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense (Dense)               (None, 128)               32896     
                                                                 
 batch_normalization (BatchN  (None, 128)              5

# Compile and Train Model

In [7]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

physical_gpus = tf.config.list_physical_devices('GPU')
logical_gpus = tf.config.list_logical_devices('GPU')
print('Physical GPUs:', physical_gpus)
print('Logical GPUs:', logical_gpus)
if logical_gpus:
    print('Training device target: GPU')
else:
    print('Training device target: CPU (GPU not detected)')

cp_callback = tf.keras.callbacks.ModelCheckpoint(model_save_path, verbose=1, save_weights_only=False)
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

model.fit(X_train, y_train, epochs=2000, batch_size=32, validation_data=(X_test, y_test), callbacks=[cp_callback, es_callback])

Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Logical GPUs: [LogicalDevice(name='/device:GPU:0', device_type='GPU')]
Training device target: GPU
Epoch 1/2000
5267/5271 [============================>.] - ETA: 0s - loss: 0.4805 - accuracy: 0.8800
Epoch 1: saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
5271/5271 [==============================] - 59s 11ms/step - loss: 0.4802 - accuracy: 0.8801 - val_loss: nan - val_accuracy: 0.9850
Epoch 2/2000
5271/5271 [==============================] - ETA: 0s - loss: 0.0702 - accuracy: 0.9807
Epoch 2: saving model to model/keypoint_classifier\keypoint_sequence_classifier.keras
5271/5271 [==============================] - 55s 10ms/step - loss: 0.0702 - accuracy: 0.9807 - val_loss: nan - val_accuracy: 0.9715
Epoch 3/2000
5267/5271 [============================>.] - ETA: 0s - loss: 0.0471 - accuracy: 0.9870
Epoch 3: saving model to model/keypoint_classifier\keypoint_sequence_classifier.ker

# Save a full SavedModel for OpenVINO conversion
Add a SavedModel export to ensure weights and graph are fully serialized.

In [8]:
saved_model_dir = 'model/keypoint_classifier/keypoint_sequence_classifier_savedmodel'
if 'model' in locals():
    os.makedirs(saved_model_dir, exist_ok=True)
    model.save(saved_model_dir)
    print(f"SavedModel exported to: {saved_model_dir}")
else:
    print("Model not found. Train the model first.")

INFO:tensorflow:Assets written to: model/keypoint_classifier/keypoint_sequence_classifier_savedmodel\assets


INFO:tensorflow:Assets written to: model/keypoint_classifier/keypoint_sequence_classifier_savedmodel\assets


SavedModel exported to: model/keypoint_classifier/keypoint_sequence_classifier_savedmodel


# Evaluate Model

In [9]:
val_loss, val_acc = model.evaluate(X_test, y_test)
print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_acc}')

1757/1757 [==============================] - 8s 5ms/step - loss: nan - accuracy: 0.9996
Validation Loss: nan, Validation Accuracy: 0.9996265172958374


# Convert to TFLite

In [10]:
model.save(model_save_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

with open(tflite_save_path, 'wb') as f:
    f.write(tflite_model)

print("TFLite model saved.")

INFO:tensorflow:Assets written to: C:\Users\lam10\AppData\Local\Temp\tmplb3126ji\assets


INFO:tensorflow:Assets written to: C:\Users\lam10\AppData\Local\Temp\tmplb3126ji\assets


TFLite model saved.


# Test Inference

In [11]:
# Test Inference
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)

# Add Flex delegate for TensorFlow ops
from tensorflow.lite.python.interpreter import load_delegate
try:
    delegate = load_delegate('libtensorflowlite_flex.so')
    interpreter = tf.lite.Interpreter(model_path=tflite_save_path, experimental_delegates=[delegate])
except:
    print("Flex delegate not available, using standard interpreter")

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]], dtype=np.float32))
interpreter.invoke()
result = interpreter.get_tensor(output_details[0]['index'])

print("Predicted:", np.argmax(result))
print("Actual:", y_test[0])

Exception ignored in: <function Delegate.__del__ at 0x000001C0A91BB130>
Traceback (most recent call last):
  File "d:\Profile\Documents\GitHub\FYP-SignLanguage\cv_hands\.venv-train310\lib\site-packages\tensorflow\lite\python\interpreter.py", line 117, in __del__
    if self._library is not None:
AttributeError: 'Delegate' object has no attribute '_library'


Flex delegate not available, using standard interpreter
Predicted: 15
Actual: 15.0
